# TV3: Bước 4 - Fact-aware Evidence Reranking & Đánh giá NLI End-to-End
## Module 9 & Pipeline Validation: Reranking → NLI Inputs → BamiBERT SOTA Inference

---

### Điểm quan trọng về Kiến trúc Đồ án:
1. **IR & IE không train NLI**: IR/IE chịu trách nhiệm sàng lọc ra **Evidence mới** có độ tin cậy và sự thật cao nhất.
2. **Tạo dữ liệu đầu vào chuẩn hóa NLI**: `(Claim, Evidence, Label)` xuất thành file CSV.
3. **Model-specific Preprocessing & Inference**:
   - Chạy mô hình NLI BamiBERT SOTA đã huấn luyện (`Exp006_PrefixPrompt_SOTA`) trên 3 thí nghiệm đối chứng:
     - **Thí nghiệm A (Gold Evidence):** Claim + Bằng chứng chuẩn (Upper-bound lý tưởng).
     - **Thí nghiệm B (BM25 Evidence):** Claim + Bằng chứng do BM25 Top 1 tìm thấy.
     - **Thí nghiệm C (BM25 + IE Reranking):** Claim + Bằng chứng do Fact-aware Reranker chọn lọc.
   - Chứng minh định lượng mức độ đóng góp của Fact-aware Reranking đối với bài toán Fact-checking!


In [1]:
import json
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("../..").resolve()
OUTPUT_DIR = PROJECT_ROOT / "data/processed/retrieval"
INPUT_PATH = OUTPUT_DIR / "evidence_features.csv"

df_feats = pd.read_csv(INPUT_PATH)

# Chuẩn hóa BM25 theo từng claim
df_feats["bm25_max"] = df_feats.groupby("claim_id")["bm25_score"].transform("max")
df_feats["bm25_norm"] = df_feats["bm25_score"] / (df_feats["bm25_max"] + 1e-6)

# Điểm số kết hợp đa tiêu chí: 0.70 BM25 + 0.30 Fact Score
ALPHA = 0.70
BETA = 0.30
df_feats["rerank_score"] = round(ALPHA * df_feats["bm25_norm"] + BETA * df_feats["fact_score"], 4)

df_feats["bm25_rank"] = df_feats["rank"]
df_feats["rerank_rank"] = df_feats.sort_values(["claim_id", "rerank_score"], ascending=[True, False]).groupby("claim_id").cumcount() + 1

output_rerank_path = OUTPUT_DIR / "reranked_evidence.csv"
df_feats.to_csv(output_rerank_path, index=False)
print(f"✓ Đã lưu bảng xếp hạng lại tại: {output_rerank_path.name}")

✓ Đã lưu bảng xếp hạng lại tại: reranked_evidence.csv


In [2]:
# Đánh giá Recall@1 và MRR trước và sau Reranking
eval_df = df_feats[df_feats["label"] != 2].copy()
n_claims = eval_df["claim_id"].nunique()

def eval_ranks(d, rank_col):
    r1 = d[(d[rank_col] == 1) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    r3 = d[(d[rank_col] <= 3) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    r5 = d[(d[rank_col] <= 5) & (d["is_gold"] == True)]["claim_id"].nunique() / n_claims
    g = d[d["is_gold"] == True]
    mrr = (1.0 / g.groupby("claim_id")[rank_col].min()).sum() / n_claims
    return r1, r3, r5, mrr

bm25_r1, bm25_r3, bm25_r5, bm25_mrr = eval_ranks(eval_df, "bm25_rank")
rerank_r1, rerank_r3, rerank_r5, rerank_mrr = eval_ranks(eval_df, "rerank_rank")

promoted = eval_df[(eval_df["bm25_rank"] > 1) & (eval_df["rerank_rank"] == 1) & (eval_df["is_gold"] == True)]

print("=" * 70)
print("SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:")
print(f"• Recall@1: BM25 = {bm25_r1*100:.2f}%  -->  Fact Reranker = {rerank_r1*100:.2f}% (Delta: {(rerank_r1-bm25_r1)*100:+.2f}%)")
print(f"• MRR:      BM25 = {bm25_mrr:.4f}  -->  Fact Reranker = {rerank_mrr:.4f} (Delta: {rerank_mrr-bm25_mrr:+.4f})")
print(f"• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: {len(promoted)} câu!")
print("=" * 70)

SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:
• Recall@1: BM25 = 89.80%  -->  Fact Reranker = 90.00% (Delta: +0.20%)
• MRR:      BM25 = 0.9297  -->  Fact Reranker = 0.9299 (Delta: +0.0002)
• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: 19 câu!


In [3]:
# Tạo các dataset đầu vào cho NLI
dev_cleaned = pd.read_csv(PROJECT_ROOT / "data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv")

# 1. Dataset A: Gold Evidence
gold_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["Evidence"].fillna(""),
    "label": dev_cleaned["labels"]
})
gold_nli.to_csv(OUTPUT_DIR / "nli_input_gold.csv", index=False)

# 2. Dataset B: BM25 Top 1 Evidence
bm25_map = df_feats[df_feats["bm25_rank"] == 1].set_index("claim_index")["retrieved_evidence"].to_dict()
bm25_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["index"].map(bm25_map).fillna(""),
    "label": dev_cleaned["labels"]
})
bm25_nli.to_csv(OUTPUT_DIR / "nli_input_bm25.csv", index=False)

# 3. Dataset C: Fact Reranked Top 1 Evidence
rerank_map = df_feats[df_feats["rerank_rank"] == 1].set_index("claim_index")["retrieved_evidence"].to_dict()
rerank_nli = pd.DataFrame({
    "claim_id": [f"dev_{i}" for i in dev_cleaned["index"]],
    "claim_index": dev_cleaned["index"],
    "statement": dev_cleaned["Statement"],
    "evidence": dev_cleaned["index"].map(rerank_map).fillna(""),
    "label": dev_cleaned["labels"]
})
rerank_nli.to_csv(OUTPUT_DIR / "nli_input_reranked.csv", index=False)

print(f"✓ Đã xuất 3 file NLI Input tại {OUTPUT_DIR}:")
print("  - nli_input_gold.csv")
print("  - nli_input_bm25.csv")
print("  - nli_input_reranked.csv")

✓ Đã xuất 3 file NLI Input tại /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/data/processed/retrieval:
  - nli_input_gold.csv
  - nli_input_bm25.csv
  - nli_input_reranked.csv


In [4]:
# ==============================================================================
# Đánh giá NLI trên mô hình BamiBERT SOTA (Exp006)
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BAMIBERT_CKPT = PROJECT_ROOT / "notebooks/models/bamibert/outputs/bamibert/experiments/Exp006_PrefixPrompt_SOTA/best_model"

print(f"Đang nạp mô hình BamiBERT SOTA từ: {BAMIBERT_CKPT.name}...")
tokenizer = AutoTokenizer.from_pretrained(BAMIBERT_CKPT)
model = AutoModelForSequenceClassification.from_pretrained(BAMIBERT_CKPT)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
model.to(device)
model.eval()

def evaluate_nli_bamibert(df_nli, batch_size=32):
    claims = [f"Tuyên bố: {str(s).strip()}" for s in df_nli["statement"]]
    evidences = [f"Bằng chứng: {str(e).strip()}" for e in df_nli["evidence"]]
    y_true = df_nli["label"].values
    y_pred = []
    
    with torch.no_grad():
        for i in range(0, len(claims), batch_size):
            inputs = tokenizer(
                claims[i:i + batch_size],
                evidences[i:i + batch_size],
                max_length=256,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)
            logits = model(**inputs).logits
            y_pred.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return acc, macro_f1

# Chạy đánh giá trên 3 nguồn bằng chứng
acc_a, f1_a = evaluate_nli_bamibert(gold_nli)
acc_b, f1_b = evaluate_nli_bamibert(bm25_nli)
acc_c, f1_c = evaluate_nli_bamibert(rerank_nli)

exp_summary = pd.DataFrame([
    {"Thí nghiệm (Experiment)": "A. Gold Evidence (Upper bound)", "Loại Bằng chứng": "Bằng chứng chuẩn gán nhãn tay", "Accuracy": f"{acc_a*100:.2f}%", "Macro-F1": f"{f1_a:.4f}"},
    {"Thí nghiệm (Experiment)": "B. BM25 Evidence", "Loại Bằng chứng": "Bằng chứng Top 1 từ BM25 thuần túy", "Accuracy": f"{acc_b*100:.2f}%", "Macro-F1": f"{f1_b:.4f}"},
    {"Thí nghiệm (Experiment)": "C. Fact-aware Reranked Evidence", "Loại Bằng chứng": "Bằng chứng Top 1 sau khi Rerank đặc trưng Fact", "Accuracy": f"{acc_c*100:.2f}%", "Macro-F1": f"{f1_c:.4f}"}
])

display(exp_summary)
exp_summary.to_csv(OUTPUT_DIR / "nli_comparison_metrics.csv", index=False)
print(f"✓ Đã lưu bảng chỉ số so sánh NLI tại: {OUTPUT_DIR / 'nli_comparison_metrics.csv'}")



Đang nạp mô hình BamiBERT SOTA từ: best_model...


,Thí nghiệm (Experiment),Loại Bằng chứng,Accuracy,Macro-F1
0,A. Gold Evidence (Upper bound),Bằng chứng chuẩn gán nhãn tay,81.05%,0.8116
1,B. BM25 Evidence,Bằng chứng Top 1 từ BM25 thuần túy,62.52%,0.6255
2,C. Fact-aware Reranked Evidence,Bằng chứng Top 1 sau khi Rerank đặc trưng Fact,62.38%,0.6242


✓ Đã lưu bảng chỉ số so sánh NLI tại: /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/data/processed/retrieval/nli_comparison_metrics.csv
